This notebook is going to try to memorize a single trajectory, but using CNN instead of MLP

The logic behind this notebook is that by using a CNN with limited receptive field, we force the network to learn the shapes rather than the literal values of the trajectory which should be slightly more challenging than literally memorizing the trajectory.

It works well

![image.png](attachment:image.png)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
pred_horizon = 132
obs_horizon = 132
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 2  # dx/dy

num_diffusion_iters = 100

## Load Dataset

In [ ]:
dataset_path = "data/gml_000000.zarr"

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon,
    action_delta=True
)

# Just grab a single stroke from indices 301 to 433
dataset.indices = dataset.indices[(dataset.indices[:, 0] == 301) & (dataset.indices[:, 1] == 433)]
dataset.indices = dataset.indices[[0] * 100]  # We want to replicate to batch multiple noise levels
print(dataset.indices.shape)

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    pin_memory=True,
    persistent_workers=True
)

# visualize data in batch
print("Num batches:          ", len(list(iter(dataloader))))
batch = next(iter(dataloader))
print("batch['obs'].shape:   ", batch['obs'].shape)
print("batch['action'].shape:", batch['action'].shape)

In [ ]:
fig, axes = plt.subplots(1, len(batch.keys()), figsize=(10, 3))
for ax, (key, val) in zip(axes, batch.items()):
    ax.plot(*val[0].T, '.-')
    ax.axis('equal')
    ax.set_title(key)

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
# # Network!
# noise_pred_net = MemorizationModel(
#     input_dim=action_dim,
#     global_cond_dim=obs_dim*obs_horizon,
#     noise_scheduler=noise_scheduler,
# )

# def init_weights(m):
#     if isinstance(m, nn.Linear):
#         nn.init.xavier_normal_(m.weight)
#         nn.init.zeros_(m.bias)
# noise_pred_net.apply(init_weights)

In [ ]:
# Network!
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=0,
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
noise_pred_net.apply(init_weights);

In [ ]:
# Test with example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# compute
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# check denoising
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

## Training

In [ ]:
num_epochs = 500

ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

if True:
    noise_pred_net.apply(init_weights)

with tqdm(range(num_epochs), desc='Epoch') as tglobal:
    all_losses = list()
    # epoch loop
    for epoch_idx in tglobal:
        epoch_loss = list()
        # batch loop
        for batch_n in dataloader:
            # Extract data
            obs_n = batch_n['obs'].to(device)
            action_n = batch_n['action'].to(device)
            B = obs_n.shape[0]
            assert obs_n.shape[1] == obs_horizon
            global_cond = obs_n.flatten(start_dim=1)
            global_cond = None

            # sample noise to add to actions
            noise = torch.randn(action_n.shape, device=device)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (B,), device=device).long()
            noisy_actions = noise_scheduler.add_noise(action_n, noise, timesteps)

            # predict the noise residual
            noise_pred = noise_pred_net(noisy_actions, timesteps, global_cond=global_cond)

            # L2 loss
            loss = nn.functional.mse_loss(noise_pred, noise)

            # optimize
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            lr_scheduler.step()

            ema.step(noise_pred_net)

            # logging
            loss_cpu = loss.item()
            epoch_loss.append(loss_cpu)
        tglobal.set_postfix(loss=np.mean(epoch_loss))
        all_losses.extend(epoch_loss)

# Weights of the EMA model is used for inference
if isinstance(noise_pred_net, MemorizationModel):
    ema_noise_pred_net = MemorizationModel(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
        noise_scheduler=noise_scheduler,
    )
else:
    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
    )
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())

# Plot the loss
plt.figure(figsize=(10, 3))
plt.semilogy(all_losses)
plt.title('Loss')

## Inference

In [ ]:
action_n_init = torch.randn((1, pred_horizon, action_dim), device=device)
# action_n_init = torch.randn((1, 80, action_dim), device=device)
history = []

action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                        global_cond=global_cond[[0]] if global_cond is not None else None,
                        log_history=history)

action_n = action_n[0].detach().cpu().numpy()
action = dataset.unnormalize_action(action_n)

In [ ]:
def integrate(action):
    obs = np.cumsum(action, axis=0)
    obs = np.concatenate([np.zeros((1, 2)), obs], axis=0)  # prepend 0
    return obs
obs = integrate(action)

In [ ]:
gt = {'action': dataset.unnormalize_action(batch['action'][0].numpy()),
      'obs': dataset.unnormalize_obs(batch['obs'][0].numpy())}
result = {'action': action,
          'obs': obs + gt['obs'][0]}

In [ ]:
fig, axes = plt.subplots(1, len(batch.keys()), figsize=(10, 3))
# fig, axes = plt.subplots(len(batch.keys()), 1, figsize=(4, 6))
for ax, (key, val) in zip(axes, gt.items()):
    print(f'{key:6s}: {val.shape} {result[key].shape}')
    # ax.plot(val, 'k.-')
    # ax.plot(result[key], 'r.-')
    if key == 'obs':
        ax.plot(*val.T, 'k.-')
        ax.plot(*result[key].T, 'r.-')
    else:
        ax.plot(val, 'k-')
        ax.plot(result[key], 'r-')
    ax.axis('equal')
    ax.set_title(key)

## Visualize Denoising

In [ ]:
def convert_to_action_obs(action_n):
    action = dataset.unnormalize_action(action_n)
    obs = integrate(action) + gt['obs'][0]
    return dict(action=action, obs=obs)

results = [convert_to_action_obs(action_n[0]) for action_n in history]

In [ ]:
# Show denoising process
fig, axes = plt.subplots(3, 3, sharex=True, figsize=(15, 6))
fig2, axes2 = plt.subplots(3, 3, sharex=True, figsize=(15, 6))
N = axes.size

samples = np.linspace(0, len(history) - 1, N).astype(int)
for ax, ax2, t, result in zip(axes.flatten(), axes2.flatten(), samples, results):
    action, obs = result['action'], result['obs']
    ax.plot(*action.T, 'r.-')
    ax.plot(*gt['action'].T, 'k.-')
    ax2.plot(*obs.T, 'r.-')
    ax2.plot(*gt['obs'].T, 'k.-')
    ax.set_title(f'Predicted Action at t={100-t}')
    ax.axis('equal')
    ax2.set_title(f'Predicted Trajectory at t={100-t}')
    ax2.axis('equal')

In [ ]:
# # Get aspect ratio

# fig, axes = plt.subplots(1, 2, figsize=(10, 3))
# axes[0].axis('equal')
# from operator import sub
# def get_aspect(ax):
#     # Total figure size
#     figW, figH = ax.get_figure().get_size_inches()
#     # Axis size on figure
#     _, _, w, h = ax.get_position().bounds
#     # Ratio of display units
#     disp_ratio = (figH * h) / (figW * w)
#     return disp_ratio
# print(get_aspect(axes[0]))

In [ ]:
# Create animation
outfolder = Path('results/gerry07')
outfolder.mkdir(exist_ok=True)

bounds = {}
# AR = 1 / 1.0929032258064517  # (10, 5)
AR = 1 / 0.655741935483871  # (10, 3)
for k in gt:
    bounds[k] = np.min(gt[k], axis=0), np.max(gt[k], axis=0)
    for result in results:
        bounds[k] = np.minimum(bounds[k][0], np.min(result[k], axis=0)), np.maximum(bounds[k][1], np.max(result[k], axis=0))
    # Correct for aspect ratio
    xs, ys = (bounds[k][0][0], bounds[k][1][0]), (bounds[k][0][1], bounds[k][1][1])
    if np.diff(xs) / np.diff(ys) > AR:
        ys = (np.mean(ys) - np.diff(xs) / AR / 2, np.mean(ys) + np.diff(xs) / AR / 2)
    else:
        xs = (np.mean(xs) - np.diff(ys) * AR / 2, np.mean(xs) + np.diff(ys) * AR / 2)
    bounds[k] = np.array(xs), np.array(ys)

def plot_frame(axes, result, gt, t):
    axes[0].clear()
    axes[1].clear()
    for ax, k in zip(axes, gt):
        ax.plot(*gt[k].T, 'k.-')
        ax.plot(*result[k].T, 'r.-')
        ax.set_title(f't = {100 - t}')
        ax.set_xlim(*bounds[k][0])
        ax.set_ylim(*bounds[k][1])
        # ax.axis('equal')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for t, result in enumerate(tqdm(results)):
    plot_frame(axes, result, gt, t)
    fig.savefig(outfolder / f'frame_{t:03d}.png')

In [ ]:
# Create video
# !ffmpeg -y -r 10 -i results/gerry07/frame_%03d.png -c:v libx264 -vf fps=25 -pix_fmt yuv420p results/gerry07/_anim.mp4
!/home/gchen328/miniconda3/bin/ffmpeg -y -r 25 -i results/gerry07/frame_%03d.png -c:v h264 -pix_fmt yuv420p results/gerry07.mp4 -hide_banner -loglevel error

In [ ]:
# Display
from IPython.display import Video
Video("results/gerry07.mp4", width=512, height=256)